In [5]:
# Core
import os, math, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# Data
import yfinance as yf

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Plots
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ===== Choose tickers =====
# OPTION A: use same tickers as your news experiment (edit this list)
# TICKERS = ["AAPL","MSFT","AMZN","GOOGL","NVDA"]

# OPTION B: mega-caps default:
TICKERS = ["AAPL","MSFT","AMZN","GOOGL","NVDA"]

START  = "2012-01-01"
END    = "2022-12-31"

SEQ_LEN = 60         # days in a sequence
THRESH  = 0.005      # 0.5% label threshold
BATCH   = 64
EPOCHS  = 100
PATIENCE= 12
LR      = 1e-3


Device: cpu


In [12]:
import yfinance as yf

test_df = yf.download("AAPL", start=START, end=END, progress=False)
print(test_df.head(), test_df.shape)


Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2012-01-03  12.359186  12.397355  12.292166  12.304188  302220800
2012-01-04  12.425605  12.462873  12.300580  12.322218  260022000
2012-01-05  12.563553  12.579181  12.402462  12.470986  271269600
2012-01-06  12.694890  12.705409  12.599318  12.615848  318292800
2012-01-09  12.674753  12.855680  12.663332  12.788058  394024400 (2768, 5)


In [15]:
def SMA(s, n): return s.rolling(n).mean()
def EMA(s, n): return s.ewm(span=n, adjust=False).mean()

def build_features(df_px):
    df = df_px.copy()
    df["Return"] = df["Close"].pct_change()
    df["LogRet"] = np.log1p(df["Return"])

    df["SMA_10"]  = SMA(df["Close"], 10)
    df["SMA_20"]  = SMA(df["Close"], 20)
    df["EMA_10"]  = EMA(df["Close"], 10)
    df["EMA_20"]  = EMA(df["Close"], 20)

    delta = df["Close"].diff()
    up, down = delta.clip(lower=0), -delta.clip(upper=0)
    roll_up = up.ewm(alpha=1/14, adjust=False).mean()
    roll_down = down.ewm(alpha=1/14, adjust=False).mean()
    rs = roll_up / (roll_down + 1e-9)
    df["RSI_14"] = 100 - (100 / (1 + rs))

    ema12 = EMA(df["Close"], 12)
    ema26 = EMA(df["Close"], 26)
    df["MACD"] = ema12 - ema26
    df["MACDsig"] = EMA(df["MACD"], 9)

    bb_mid = SMA(df["Close"], 20)
    bb_std = df["Close"].rolling(20).std()
    df["BB_up"] = bb_mid + 2*bb_std
    df["BB_dn"] = bb_mid - 2*bb_std

    return df  # don't dropna here

def label_direction(df, thr=0.005):
    r = df["Close"].pct_change().shift(-1).to_numpy().flatten()  # force 1D
    lab = np.full(len(df), 1)
    lab[r >  thr] = 2
    lab[r < -thr] = 0
    df["Label"] = lab
    return df

FEATURES = [
    "Open","High","Low","Close","Volume",
    "Return","LogRet","SMA_10","SMA_20","EMA_10","EMA_20",
    "RSI_14","MACD","MACDsig","BB_up","BB_dn"
]

required_cols = ["Open", "High", "Low", "Close", "Volume"]

frames = []
for t in TICKERS:
    # Download with multi-index columns
    px = yf.download(t, start=START, end=END, progress=False)
    if px is None or px.empty:
        print(f"[WARN] No data for {t}, skipping.")
        continue

    # If columns are MultiIndex, take only the first level
    if isinstance(px.columns, pd.MultiIndex):
        px.columns = [c[0] for c in px.columns]  # ('close','aapl') → 'close'

    # Normalize to Title case
    px.columns = [c.capitalize() for c in px.columns]

    # Keep only OHLCV
    if not set(required_cols).issubset(px.columns):
        print(f"[WARN] Missing OHLCV columns for {t}: have {list(px.columns)}")
        continue
    px = px[required_cols].dropna()
    if px.empty:
        print(f"[WARN] Empty after cleaning for {t}, skipping.")
        continue

    # Feature engineering
    df = build_features(px)
    df = label_direction(df, thr=THRESH)

    # Ensure all engineered features exist
    if not set(FEATURES + ["Label"]).issubset(df.columns):
        print(f"[WARN] Missing engineered features for {t}, skipping.")
        continue
    df = df.dropna(subset=FEATURES + ["Label"]).copy()
    if df.empty:
        print(f"[WARN] No rows left after dropna for {t}, skipping.")
        continue

    df["Ticker"] = t
    df["Date"] = df.index
    frames.append(df)

assert frames, "No price data gathered."
df_all = pd.concat(frames, axis=0).reset_index(drop=True)
print(df_all.shape, df_all[["Ticker", "Date", "Label"]].head())

(13745, 19)   Ticker       Date  Label
0   AAPL 2012-01-31      1
1   AAPL 2012-02-01      1
2   AAPL 2012-02-02      2
3   AAPL 2012-02-03      2
4   AAPL 2012-02-06      2


In [16]:
# global chronological split by date for fair comparison
dates_sorted = np.sort(df_all["Date"].unique())
d_train = dates_sorted[int(0.0*len(dates_sorted)) : int(0.70*len(dates_sorted))][-1]
d_val   = dates_sorted[int(0.70*len(dates_sorted)) : int(0.85*len(dates_sorted))][-1]

train_df = df_all[df_all["Date"] <= d_train].copy()
val_df   = df_all[(df_all["Date"] > d_train) & (df_all["Date"] <= d_val)].copy()
test_df  = df_all[df_all["Date"] > d_val].copy()

print("Split dates:")
print("  Train  ≤", pd.to_datetime(d_train).date())
print("  Val    >", pd.to_datetime(d_train).date(), "and ≤", pd.to_datetime(d_val).date())
print("  Test   >", pd.to_datetime(d_val).date())
print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Label balance (train):", train_df["Label"].value_counts(normalize=True).sort_index())


Split dates:
  Train  ≤ 2019-09-23
  Val    > 2019-09-23 and ≤ 2021-05-12
  Test   > 2021-05-12
Sizes: 9620 2060 2065
Label balance (train): Label
0    0.303534
1    0.328482
2    0.367983
Name: proportion, dtype: float64


In [17]:
scaler = StandardScaler().fit(train_df[FEATURES].values)

def create_sequences_per_ticker(df_split, seq_len=SEQ_LEN):
    Xs, ys = [], []
    for t in df_split["Ticker"].unique():
        g = df_split[df_split["Ticker"] == t].sort_values("Date")
        if len(g) < seq_len + 1: 
            continue
        X_scaled = scaler.transform(g[FEATURES].values)
        y = g["Label"].values.astype(np.int64)
        for i in range(seq_len, len(g)):
            Xs.append(X_scaled[i-seq_len:i])
            ys.append(y[i])
    if len(Xs) == 0:
        return np.empty((0, seq_len, len(FEATURES)), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int64)

X_tr, y_tr = create_sequences_per_ticker(train_df, SEQ_LEN)
X_va, y_va = create_sequences_per_ticker(val_df,   SEQ_LEN)
X_te, y_te = create_sequences_per_ticker(test_df,  SEQ_LEN)

print("Shapes:")
print("  Train:", X_tr.shape, y_tr.shape)
print("  Val:  ", X_va.shape, y_va.shape)
print("  Test: ", X_te.shape, y_te.shape)
print("Label balance Test:", np.bincount(y_te)/len(y_te))


Shapes:
  Train: (9320, 60, 16) (9320,)
  Val:   (1760, 60, 16) (1760,)
  Test:  (1765, 60, 16) (1765,)
Label balance Test: [0.41133144 0.21189802 0.37677054]


In [18]:
class SeqDS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i])

train_loader = DataLoader(SeqDS(X_tr, y_tr), batch_size=BATCH, shuffle=True, drop_last=True)
val_loader   = DataLoader(SeqDS(X_va, y_va), batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(SeqDS(X_te, y_te), batch_size=BATCH, shuffle=False)

class PriceMovementLSTM(nn.Module):
    def __init__(self, n_features, hidden=128, layers=2, dropout=0.25, bidirectional=True, n_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden, num_layers=layers,
            batch_first=True, dropout=dropout if layers>1 else 0.0,
            bidirectional=bidirectional
        )
        out_dim = hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Dropout(dropout),
            nn.Linear(out_dim, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        out, _ = self.lstm(x)           # [B, T, H]
        last = out[:, -1, :]            # last timestep
        logits = self.head(last)        # [B, C]
        return logits

model = PriceMovementLSTM(n_features=len(FEATURES)).to(DEVICE)
print(model)


PriceMovementLSTM(
  (lstm): LSTM(16, 128, num_layers=2, batch_first=True, dropout=0.25, bidirectional=True)
  (head): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Dropout(p=0.25, inplace=False)
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)


In [19]:
# Class weights to handle imbalance a bit (optional but helpful)
class_counts = np.bincount(y_tr, minlength=3).astype(np.float32)
class_weights = (class_counts.sum() / (class_counts + 1e-9))
class_weights = class_weights / class_weights.mean()
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, preds_all, labels_all = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        preds_all.append(logits.detach().cpu().numpy())
        labels_all.append(yb.detach().cpu().numpy())
    preds = np.argmax(np.vstack(preds_all), axis=1)
    labels= np.hstack(labels_all)
    acc = (preds == labels).mean()
    f1m = f1_score(labels, preds, average="macro")
    return total_loss/len(loader.dataset), acc, f1m

best_val, patience = 1e9, 0
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, True)
    va_loss, va_acc, va_f1 = run_epoch(val_loader, False)
    scheduler.step(va_loss)
    if va_loss < best_val - 1e-5:
        best_val = va_loss; patience = 0
        torch.save(model.state_dict(), "best_price_cls_lstm.pth")
    else:
        patience += 1
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | "
              f"train loss {tr_loss:.4f} acc {tr_acc:.3f} f1 {tr_f1:.3f}  |  "
              f"val loss {va_loss:.4f} acc {va_acc:.3f} f1 {va_f1:.3f}")
    if patience >= PATIENCE:
        print("Early stopping.")
        break

model.load_state_dict(torch.load("best_price_cls_lstm.pth", map_location=DEVICE))
model.eval()


Class weights: [1.0929607 1.0047557 0.9022835]
Epoch 001 | train loss 1.0958 acc 0.365 f1 0.357  |  val loss 1.0818 acc 0.416 f1 0.271
Epoch 005 | train loss 1.0823 acc 0.367 f1 0.343  |  val loss 1.0705 acc 0.368 f1 0.338
Epoch 010 | train loss 1.0785 acc 0.379 f1 0.369  |  val loss 1.0743 acc 0.345 f1 0.331
Epoch 015 | train loss 1.0787 acc 0.374 f1 0.346  |  val loss 1.0776 acc 0.323 f1 0.277
Epoch 020 | train loss 1.0708 acc 0.391 f1 0.377  |  val loss 1.0691 acc 0.381 f1 0.337
Early stopping.


PriceMovementLSTM(
  (lstm): LSTM(16, 128, num_layers=2, batch_first=True, dropout=0.25, bidirectional=True)
  (head): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Dropout(p=0.25, inplace=False)
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)

In [20]:
def infer(loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE))
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            preds_all.append(preds)
            labels_all.append(yb.numpy())
    return np.hstack(preds_all), np.hstack(labels_all)

y_pred, y_true = infer(test_loader)
acc = accuracy_score(y_true, y_pred)
f1m = f1_score(y_true, y_pred, average="macro")

print(f"TEST — acc {acc:.3f}  macroF1 {f1m:.3f}")
print(classification_report(y_true, y_pred, target_names=["Down","Neutral","Up"]))


TEST — acc 0.388  macroF1 0.263
              precision    recall  f1-score   support

        Down       0.38      0.13      0.19       726
     Neutral       0.32      0.03      0.06       374
          Up       0.39      0.87      0.54       665

    accuracy                           0.39      1765
   macro avg       0.36      0.34      0.26      1765
weighted avg       0.37      0.39      0.29      1765



In [21]:
# Rebuild per-ticker test sequences the same way to slice predictions cleanly
def per_ticker_counts(df_split):
    counts = {}
    for t in df_split["Ticker"].unique():
        g = df_split[df_split["Ticker"] == t].sort_values("Date")
        n = max(0, len(g) - SEQ_LEN)
        counts[t] = n
    return counts

counts = per_ticker_counts(test_df)
idx = 0
perf = {}
for t in test_df["Ticker"].unique():
    n = counts.get(t, 0)
    if n <= 0: 
        continue
    preds = y_pred[idx:idx+n]
    true  = y_true[idx:idx+n]
    perf[t] = 100.0 * (preds == true).mean()
    idx += n

print("Per-ticker test accuracy:")
for k, v in sorted(perf.items(), key=lambda x: x[1], reverse=True):
    print(f"  {k}: {v:.1f}%")


Per-ticker test accuracy:
  NVDA: 43.9%
  AAPL: 39.1%
  AMZN: 38.0%
  GOOGL: 36.8%
  MSFT: 36.0%
